# Pipeline Deployment

# Pipeline Deploymet

In this demo, we will show how to use a model as part of a data pipeline for inference. In the first section of the demo, we will prepare data and perform some basic feature engineering. Then, we will fit and register the model to model registry. Please note that these two steps are already covered in other courses and they are not the main focus of this demo. In the last section, which is the main focus of this demo, we will create a Delta Live Tables (DLT) pipeline and use the registered model as part of the pipeline.

### Learning Objectives:

*By the end of this demo, you will be able to:*

* Describe steps for deploying a model with in a pipeline.
* Develop a simple Delta Live Tables pipeline that performs batch inference in its final step.

## Requirements

Please review the following requirements before starting the lesson:
* To run this notebook, you need to use one of the following Databricks runtime(s): **{{supported_dbrs}}**

## Classroom Setup

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo. Execute the following cell:

In [0]:
# %run ../Includes/Classroom-Setup-01

## Other Conventions:

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
#print(f"Username:          {DA.username}")
#print(f"Catalog Name:      {DA.catalog_name}")
#print(f"Schema Name:       {DA.schema_name}")
#print(f"Working Directory: {DA.paths.working_dir}")
#print(f"User DB Location:  {DA.paths.datasets}")

## Data Preparation

For this demonstration, we will utilize a fictional dataset from a Telecom Company, which includes customer information. This dataset encompasses **customer demographics**, including gender, as well as internet subscription details such as subscription plans and payment methods.

After load the dataset, we will perform simple **data cleaning and feature selection**.

In the final step, we will split the dataset to **features** and **response** sets.

In [0]:


# 0. Instalar a biblioteca do Kaggle via comando de terminal no cluster
# (O caractere '!' permite rodar comandos shell diretamente do notebook)
!pip install -q kaggle

# 1. Criar uma pasta local temporária no driver e baixar o dataset do link informado
!mkdir -p /tmp/kaggle_data
!kaggle datasets download -d blastchar/telco-customer-churn -p /tmp/kaggle_data --unzip

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
License(s): copyright-authors
100%|████████████████████████████████████████| 172k/172k [00:00<00:00, 5.63MB/s]



In [0]:
from pyspark.sql.functions import col



import os

# 2. Configurar as credenciais do Kaggle no ambiente do cluster
os.environ['KAGGLE_USERNAME'] = "fabienecasamento" #"SEU_USUARIO_AQUI"
with open('/Workspace/Users/fabieneaulas@gmail.com/ML_DEPLOY/chave_kaggle.txt', 'r') as leitura_chave:
    kaggle_key = leitura_chave.read().strip()
# Extrair os dados antes do ';|' na variável kaggle_key
kaggle_key_before_delimiter = kaggle_key.split(';')[0]


os.environ['KAGGLE_KEY'] = kaggle_key_before_delimiter #"SUA_CHAVE_AQUI" # token databricks

# dataset path
#dataset_p_telco = f"{DA.paths.datasets}/telco/telco-customer-churn.csv"





In [0]:

# 4. Copiar o arquivo do /tmp para o Workspace (serverless requer /Workspace ou Volumes)
import shutil
import os

workspace_dir = "/Workspace/Users/fabieneaulas@gmail.com/ML_DEPLOY/kaggle_data"
os.makedirs(workspace_dir, exist_ok=True)
shutil.copy("/tmp/kaggle_data/WA_Fn-UseC_-Telco-Customer-Churn.csv", f"{workspace_dir}/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# 5. Abrir e ler o arquivo utilizando o PySpark
# O arquivo baixado do Kaggle adota o nome padrão original da IBM ("WA_Fn-UseC_-...")
dataset_path = f"{workspace_dir}/WA_Fn-UseC_-Telco-Customer-Churn.csv"

dataset_p_telco = spark.read.csv(
    dataset_path, 
    inferSchema=True, 
    header=True, 
    multiLine=True, 
    escape='"'
)

# 6. Visualizar o DataFrame carregado com sucesso no Spark
display(dataset_p_telco.show(8))

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|         No|    

In [0]:
# Dataset specs
primary_key = "customerID"
response = "Churn"
features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"] # Keeping numerical only for simplicity and demo purposes

# Read dataset (and drop nan), leitura novamente
from pyspark.sql.functions import trim

telco_df = spark.read.csv(dataset_path, inferSchema=True, header=True, multiLine=True, escape='"')\
    .filter(trim(col("TotalCharges")) != "")\
    .withColumn("TotalCharges", col("TotalCharges").cast('double'))\
    .na.drop(how='any')


#Perfect! I've identified the root cause. The TotalCharges column contains 11 rows with empty string values (whitespace) that cannot be cast to double. These are customers with tenure=0 (new customers with no total charges yet).


# Separate features and ground-truth
features_df = telco_df.select(primary_key, *features)
response_df = telco_df.select(primary_key, response)

# Train a sklearn Decision Tree Classification model
# Covert data to pandas dataframes
X_train_pdf = features_df.drop(primary_key).toPandas()
Y_train_pdf = response_df.drop(primary_key).toPandas()

for col in X_train_pdf.select_dtypes("int32"):
    X_train_pdf[col] = X_train_pdf[col].astype("double")

In [0]:
print(X_train_pdf.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   SeniorCitizen   7032 non-null   float64
 1   tenure          7032 non-null   float64
 2   MonthlyCharges  7032 non-null   float64
 3   TotalCharges    7032 non-null   float64
dtypes: float64(4)
memory usage: 219.9 KB
None


In [0]:
# página 17

## Model Preparation

**Note:** This section is not the main focus of this course. We are just repeating the model development and registration process here.

### Setup Model Registry with UC

Before we start model deployment, we need to fit and register a model. In this demo, **we will log models to Unity Catalog**, which means first we need to setup the **MLflow Model Registry URI**.

In [0]:
import mlflow

# Point to UC model registry
mlflow.set_registry_uri("databricks-uc")
client = mlflow.MlflowClient()

def get_latest_model_version(model_name):
    """Helper function to get latest model version"""
    model_version_infos = client.search_model_versions(f"name = '%s'"% model_name)
    return max([model_version_info.version for model_version_info in model_version_infos])

## Fit and Register a Model with UC

In [0]:
from sklearn.tree import DecisionTreeClassifier
from mlflow.models import infer_signature

# Use 3-level namespace for model name
catalog_name= 'workspace'
schema_name = 'default'

model_name = f"{catalog_name}.{schema_name}.ml_model_auladeploy"

# model to use for classification
clf = DecisionTreeClassifier(max_depth=4, random_state=10)

with mlflow.start_run(run_name="Model-Deployment demo") as mlflow_run:
    
    # Enable automatic logging of input samples, metrics, parameters, and models
    mlflow.sklearn.autolog(
        log_input_examples=True,
        log_models=False,
        log_post_training_metrics=True,
        silent=True
    )
    
    clf.fit(X_train_pdf, Y_train_pdf)
    
    # Log model and push to registry
    signature = infer_signature(X_train_pdf, Y_train_pdf)
    mlflow.sklearn.log_model(
        clf,
        artifact_path="decision_tree",
        signature=signature,
        registered_model_name=model_name
    )
    
    # Set model alias
    client.set_registered_model_alias(model_name, "DLT", get_latest_model_version(model_name))

2026/06/07 21:55:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-0b69ef93-5247.cloud.databricks.com/ml/experiments/2014947904566416/models/m-817622c400ce4d4995199b0faa5bd455?o=7474657872577658
Successfully registered model 'workspace.default.ml_model_auladeploy'.


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.ml_model_auladeploy': https://dbc-0b69ef93-5247.cloud.databricks.com/explore/data/models/workspace/default/ml_model_auladeploy/version/1?o=7474657872577658


## Configure Pipeline to Run Batch Inference

Now that our model is registered and ready, we can move on to the most important part; using the model for inference inside a pipeline.

**Note:** The DLT pipeline is already defined in `3.1.b` notebook.

**Note:** If you want to learn more about DLT please check out `Data Engineerign with Databricks` (Data Pipeline with Delta Live Tables).

### Config Variables

While defining the DLT pipeline, you will need to use the following variables. Run the code block below first. Then, use the output in the next section while creating the pipeline.

In [0]:
print(f"mlpipeline.bronze_dataset_path: {dataset_p_telco}")
print(f"mlpipeline.model_name: {model_name}")

mlpipeline.bronze_dataset_path: DataFrame[customerID: string, gender: string, SeniorCitizen: int, Partner: string, Dependents: string, tenure: int, PhoneService: string, MultipleLines: string, InternetService: string, OnlineSecurity: string, OnlineBackup: string, DeviceProtection: string, TechSupport: string, StreamingTV: string, StreamingMovies: string, Contract: string, PaperlessBilling: string, PaymentMethod: string, MonthlyCharges: double, TotalCharges: string, Churn: string]
mlpipeline.model_name: workspace.default.ml_model_auladeploy


## Create the DLT Pipeline

to create a pipeline, follow these steps:

Got to Delta Live Tables from the left menu


In [0]:
# não achei o Delta Live tables no lado esquerdo do databricks (abaixo do Data Engineering), no databricks model


In [0]:
# página 30